# Build B — Threat Intel RAG Digest (Kaggle GPU run)

Prerequisites: enable GPU (Settings → Accelerator → GPU) and enable internet
access for this notebook. The repo (code + corpus) is public, so no token or secret is needed.


In [ ]:
import sys
REPO_URL = "https://github.com/av4874/RAG_threatIntel.git"

!{sys.executable} -m pip install -q transformers accelerate torch sentence-transformers faiss-cpu
!git clone {REPO_URL} /kaggle/working/repo

# Editable installs (pip install -e) write a .pth file that Python only
# reads at interpreter startup -- since this kernel is already running,
# a mid-session editable install is never picked up. Add the source
# directory to sys.path directly instead, which works immediately.
sys.path.insert(0, "/kaggle/working/repo/src")


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)


class QwenLLMClient:
    def generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        output_ids = model.generate(inputs, max_new_tokens=512, do_sample=True, temperature=0.2)
        generated = output_ids[0][inputs.shape[-1]:]
        return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
from pathlib import Path
from threat_digest.pipeline import run_pipeline

digest_path = run_pipeline(
    corpus_dir=Path("/kaggle/working/repo/corpus"),
    output_dir=Path("/kaggle/working/output"),
    llm_client=QwenLLMClient(),
    k=5,
)
print(f"Digest written to: {digest_path}")
print(digest_path.read_text())


## Validate before trusting the digest

Before treating any entry as real, open `output/audit.jsonl` and check, for a
few items: do the retrieved chunks actually support the summary and risk
score? If the model is inventing detail not present in the retrieved text,
that's a grounding failure — see the design spec's Testing/Validation section.